In [23]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


#RUN THIS BLOCK AND READ THE PRINT STATEMENT TO LEARN ABOUT DATA STRUCTURE IN REGION H5 FILE

#UPDATE FILE PATH FOR FULL REGION 
file_path = "practice_halo_catalog.hdf5"

#PRINT ALL HALO NAMES
PRINT_ALL_HALO_NAMES = False


print("\nOpening HDF5 file...\n")

with h5py.File(file_path, "r") as f:

    units = f["units"].attrs if "units" in f else {}

    def get_unit(name):
        val = units.get(name, "")

        if isinstance(val, bytes):
            val = val.decode()

        if val in ["unitary", "dimensionless"]:
            return ""
        if val == "g":
            return "g (grams)"

        return val

    def get_shape_meaning(name, shape):
        if len(shape) == 1:
            return f"{shape[0]} values"

        if len(shape) == 2:
            if shape[1] == 3:
                return f"{shape[0]} entries × (x, y, z)"
            if "SED" in name:
                return f"{shape[0]} objects × {shape[1]} wavelength bins"
            return f"{shape[0]} × {shape[1]} array"

        if len(shape) == 3:
            return f"{shape[0]} × {shape[1]} × {shape[2]} grid"

        return "multi-dimensional data"

    keys = list(f.keys())

    halo_names = [k for k in keys if k.startswith("halo_")]
    other_keys = [k for k in keys if not k.startswith("halo_")]

    print("Top-level structure:")
    print(f"  Total entries: {len(keys)}")
    print(f"  Number of halos: {len(halo_names)}")
    print(f"  Example halos: {halo_names[:5]}")

    if len(other_keys) > 0:
        print(f"  Other entries: {other_keys}")
        if "units" in other_keys:
            print("    → 'units' contains the physical units")

    print()

    # USER GUIDANCE PRINT STATEMENTS
    print("Options:")
    print("  → Set PRINT_ALL_HALO_NAMES = True to print all halo names")

    #print all halo names boolean
    if PRINT_ALL_HALO_NAMES:
        print(" ALL HALO NAMES ")

        for i, name in enumerate(halo_names):
            print(name)

            # progress for large files
            if (i + 1) % 200 == 0:
                print(f"... printed {i+1} halos ...")

        print(f"\nTotal halos: {len(halo_names)}\n")
    else:
        print(f"(Set PRINT_ALL_HALO_NAMES = True to list all {len(halo_names)} halos)\n")

    example_halo = halo_names[0]
    halo = f[example_halo]

    print(f" HALO SUMMARY: {example_halo}")

    print(f"{'Field':<40} {'Shape':<20} {'Type':<12} {'Unit':<25} {'Meaning'}")
    print("-" * 140)

    dataset_count = 0

    for key in sorted(halo.keys()):
        obj = halo[key]

        if isinstance(obj, h5py.Dataset):
            dataset_count += 1

            shape = obj.shape
            dtype = str(obj.dtype)

            unit = get_unit(key)
            if unit == "":
                unit = "-"

            meaning = get_shape_meaning(key, shape)

            print(f"{key:<40} {str(shape):<20} {dtype:<12} {unit:<25} {meaning}")
            print()

    print(f"Total datasets: {dataset_count}\n")

    print(" HALO ATTRIBUTES (METADATA)")

    print(f"{'Attribute':<40} {'Value':<25} {'Unit'}")
    print("-" * 130)

    attr_count = 0

    for attr in sorted(halo.attrs.keys()):
        attr_count += 1

        value = halo.attrs[attr]

        unit = get_unit(attr)
        if unit == "":
            unit = "-"

        print(f"{attr:<40} {str(value):<25} {unit}")
        print()

    print(f"Total attributes: {attr_count}\n")

print("\nDone! File explored.\n")


Opening HDF5 file...

Top-level structure:
  Total entries: 12
  Number of halos: 11
  Example halos: ['halo_0', 'halo_1', 'halo_13', 'halo_2', 'halo_3']
  Other entries: ['units']
    → 'units' contains the physical units

Options:
  → Set PRINT_ALL_HALO_NAMES = True to print all halo names
(Set PRINT_ALL_HALO_NAMES = True to list all 11 halos)

 HALO SUMMARY: halo_0
Field                                    Shape                Type         Unit                      Meaning
--------------------------------------------------------------------------------------------------------------------------------------------
angular_momentum                         (3,)                 float32      Mpc*Msun*km/(h**2*s)      3 values

bh_particle_IDs                          (7,)                 float64      -                         7 values

gas_mass                                 (303658,)            float64      g (grams)                 303658 values

pop2_metallicity_fraction               